# Traffic Intersection End-to-End Tracking & Performance Analysis (Colab Version)

This notebook implements the complete vehicle detection, tracking, camera calibration, and performance metrics pipeline of the SimJam Computer Vision codebase. 

### How to use on Google Colab:
1. **Setup**: Run **Step 0: Install Dependencies** to install `ultralytics` and `supervision` libraries.
2. **Upload Video**: Open the files tab on the left sidebar and drag-and-drop your video (e.g. `vehicles.mp4`) and optionally your custom YOLO weights (`yolo11l.pt`).
3. **Camera Calibration & Drawing**: Run **Step 2** to extract a frame, display it with a coordinate grid, enter coordinates for the Region of Interest (ROI) and Lanes, and preview your drawing.
4. **YOLO Detection & Tracking**: Run **Step 3** to run YOLO and ByteTrack on the video, estimating speeds and outputting the vehicle track coordinates (`vehicle_tracks_xy.csv`).
5. **Performance Analysis**: Run **Step 4** to execute the lane-assignment geometry check and calculate traffic performance parameters (throughput, average speed, delay, Level of Service) exported to `lane_metrics.csv`.
6. **Visual Analysis**: Run **Step 5** to view interactive performance charts.

## Step 0: Install Dependencies

In [ ]:
# Install YOLO (ultralytics) and supervision library
!pip install -q ultralytics supervision

## Step 1: Imports & Core Pipeline Logic

In [ ]:
import os
import sys
import csv
import json
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import supervision as sv
from ultralytics import YOLO

In [ ]:
# --- BGR Colors and Lane Colors from calib_and_track_ui.py ---
GREY   = (128, 128, 128)
YELLOW = (0, 255, 255)
WHITE  = (255, 255, 255)
BLACK  = (0, 0, 0)

LANE_COLORS = [
    (255, 0, 0),    # Blue
    (0, 255, 0),    # Green
    (0, 0, 255),    # Red
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 165, 255),  # Orange
    (128, 0, 128),  # Purple
    (0, 128, 128),  # Olive
]

# --- PlaneMapper and Tracking Helpers from calib_and_track_ui.py ---
class PlaneMapper:
    def __init__(self, src_quad: np.ndarray, dst_quad: np.ndarray) -> None:
        self._H = cv2.getPerspectiveTransform(src_quad.astype(np.float32), dst_quad.astype(np.float32))

    def warp_points(self, pts_xy: np.ndarray) -> np.ndarray:
        if pts_xy is None or len(pts_xy) == 0:
            return np.zeros((0, 2), dtype=np.float32)
        pts = pts_xy.reshape(-1, 1, 2).astype(np.float32)
        warped = cv2.perspectiveTransform(pts, self._H)
        return warped.reshape(-1, 2)


def mean_speed_kmh(track_pts, fps):
    if len(track_pts) < 2:
        return None
    (x0, y0) = track_pts[0]
    (x1, y1) = track_pts[-1]
    d_units = float(np.hypot(x1 - x0, y1 - y0))
    dt = len(track_pts) / fps
    if dt <= 0:
        return None
    return (d_units / dt) * 3.6


def iou_xyxy(a, b):
    xA = max(a[0], b[0]); yA = max(a[1], b[1])
    xB = min(a[2], b[2]); yB = min(a[3], b[3])
    inter_w = max(0.0, xB - xA); inter_h = max(0.0, yB - yA)
    inter = inter_w * inter_h
    if inter == 0:
        return 0.0
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter
    return inter / max(union, 1e-6)


def suppress_track_dupes(dets: sv.Detections, iou=0.85) -> sv.Detections:
    if len(dets) <= 1:
        return dets
    boxes = dets.xyxy
    confs = dets.confidence if dets.confidence is not None else np.ones((len(dets),), dtype=float)
    order = np.argsort(-confs)
    keep_mask = np.ones((len(dets),), dtype=bool)
    for i_idx in range(len(order)):
        i = order[i_idx]
        if not keep_mask[i]:
            continue
        for j_idx in range(i_idx + 1, len(order)):
            j = order[j_idx]
            if not keep_mask[j]:
                continue
            ti = None if dets.tracker_id is None else dets.tracker_id[i]
            tj = None if dets.tracker_id is None else dets.tracker_id[j]
            if ti is None or tj is None:
                continue
            if iou_xyxy(boxes[i], boxes[j]) >= iou:
                keep_mask[j] = False
    return dets[keep_mask]


def make_polygon_mask(img_shape, polygon_xy):
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [polygon_xy.astype(np.int32)], 255)
    return mask


def filter_detections_to_polygon(dets: sv.Detections, polygon_xy: np.ndarray, anchor: sv.Position = sv.Position.CENTER):
    if len(dets) == 0:
        return dets, np.zeros((0,), dtype=bool)
    anchors = dets.get_anchors_coordinates(anchor=anchor).astype(np.float32)
    poly = polygon_xy.astype(np.float32)
    inside = np.array([cv2.pointPolygonTest(poly, (float(x), float(y)), False) >= 0 for (x, y) in anchors], dtype=bool)
    return dets[inside], inside


def stylize_display(base_frame, polygon_xy, lanes, ln_px, blur_outside=True):
    mask = make_polygon_mask(base_frame.shape, polygon_xy)
    if blur_outside:
        overlay = base_frame.copy()
        cv2.fillPoly(overlay, [polygon_xy.astype(np.int32)], color=(0, 255, 255))
        tinted = base_frame.copy()
        cv2.addWeighted(overlay, 0.15, tinted, 0.85, 0, tinted)
        blurred = cv2.GaussianBlur(tinted, (31, 31), 0)
        display = blurred.copy()
        mask3 = cv2.merge([mask, mask, mask])
        display[mask3 == 255] = tinted[mask3 == 255]
    else:
        display = base_frame.copy()

    for lane_idx, lane in enumerate(lanes):
        color = LANE_COLORS[lane_idx % len(LANE_COLORS)]
        lane_pts = lane.astype(np.int32)
        overlay = display.copy()
        cv2.fillPoly(overlay, [lane_pts], color)
        cv2.addWeighted(overlay, 0.15, display, 0.85, 0, display)
        cv2.polylines(display, [lane_pts], True, color, max(1, ln_px - 1))

    cv2.polylines(display, [polygon_xy.astype(np.int32)], True, (0, 255, 255), ln_px)
    return display, mask


# --- Geometry, Dataclasses, and CVIntersectionAnalyzer from cv_intersection_performance_analysis.py ---
STOPPED_SPEED_THRESHOLD = 0.5  # m/s (< 1.8 km/h considered stopped)
QUEUE_SPEED_THRESHOLD = 1.0    # m/s
FREE_FLOW_PERCENTILE = 85
MEASUREMENT_START_TIME = 0.0
MIN_TRACK_POINTS = 3
MIN_FRAMES_FOR_COUNTING = 10  # Minimum frames required to count a vehicle
MIN_DETECTION_TIME = 0.5  # Minimum detection time in seconds to count a vehicle


# ============================================================================
# GEOMETRY FUNCTIONS FOR LANE ASSIGNMENT
# ============================================================================

def point_in_polygon(x, y, poly_points):
    """
    Check if point (x, y) is inside polygon defined by poly_points.
    Uses ray casting algorithm.
    
    Args:
        x, y: Point coordinates
        poly_points: List of tuples [(x1,y1), (x2,y2), ...]
    
    Returns:
        bool: True if point is inside polygon
    """
    n = len(poly_points)
    inside = False
    
    p1x, p1y = poly_points[0]
    for i in range(1, n + 1):
        p2x, p2y = poly_points[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    
    return inside


def assign_lane_to_point(img_x, img_y, lane_polygons):
    """
    Assign a lane ID to a trajectory point based on its image coordinates.
    
    Args:
        img_x, img_y: Image coordinates of the point
        lane_polygons: Dict of {lane_id: [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]}
    
    Returns:
        int or None: Lane ID if point is inside a lane polygon, None otherwise
    """
    for lane_id, polygon in lane_polygons.items():
        if point_in_polygon(img_x, img_y, polygon):
            return lane_id
    return None


# ============================================================================
# CORE ANALYSIS FUNCTIONS
# ============================================================================

def calculate_LOS_from_delay(control_delay):
    """HCM 2016 Table 19-8"""
    if control_delay <= 10: return 'A'
    elif control_delay <= 20: return 'B'
    elif control_delay <= 35: return 'C'
    elif control_delay <= 55: return 'D'
    elif control_delay <= 80: return 'E'
    else: return 'F'


@dataclass
class VehicleMetrics:
    vehicle_id: int
    lane_id: int
    first_seen: float
    last_seen: float
    total_time: float
    total_distance: float
    avg_speed: float
    max_speed: float
    stopped_delay: float      # CHANGED: Time spent stopped (for stopped delay calculation)
    control_delay: float      # CHANGED: Total control delay (actual - free flow)
    queue_time: float
    free_flow_time: float
    completed: bool
    num_stops: int
    trajectory_points: int


@dataclass
class LaneGroupMetrics:
    lane_id: int
    throughput: int
    n_vehicles_completed: int
    n_vehicles_total: int
    avg_control_delay: float      # CHANGED: Average control delay
    avg_stopped_delay: float      # CHANGED: Average stopped delay
    avg_speed: float
    measured_flow: float
    los: str
    completion_rate: float


def convert_to_python_types(obj):
    """Convert numpy types to Python native types for JSON serialization"""
    if isinstance(obj, (np.int64, np.int32, np.int16, np.int8)):
        return int(obj)
    elif isinstance(obj, (np.float64, np.float32, np.float16)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {convert_to_python_types(k): convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_to_python_types(item) for item in obj)
    return obj


# ============================================================================
# CV INTERSECTION ANALYZER
# ============================================================================

class CVIntersectionAnalyzer:
    
    def __init__(self, trajectory_file: str, 
                 free_flow_speed_kmh: float = 50.0,
                 verbose: bool = True):
        """
        Args:
            trajectory_file: Path to CSV with vehicle tracks
            free_flow_speed_kmh: Assumed free-flow speed for delay calculation (default 50 km/h)
            verbose: Print analysis info to console
        """
        self.trajectory_file = trajectory_file
        self.free_flow_speed_mps = free_flow_speed_kmh / 3.6  # Convert to m/s
        self.verbose = verbose
        
        if self.verbose:
            print(f"\n{'='*80}")
            print("CV INTERSECTION ANALYSIS - MODIFIED")
            print(f"{'='*80}")
            print(f"File: {os.path.basename(trajectory_file)}")
            print(f"Free-flow speed: {free_flow_speed_kmh:.1f} km/h ({self.free_flow_speed_mps:.2f} m/s)")
            print(f"{'='*80}\n")
        
        self.df = pd.read_csv(trajectory_file)
        
        # Filter out any points with negative lane_id
        if 'lane_id' in self.df.columns:
            points_before = len(self.df)
            self.df = self.df[self.df['lane_id'] >= 0].copy()
            points_after = len(self.df)
            if points_before != points_after and self.verbose:
                print(f"[INFO] Filtered out {points_before - points_after} points with invalid lane_id")
        
        self._preprocess_data()
        
        self.vehicle_metrics: Dict[int, VehicleMetrics] = {}
        self.lane_metrics: Dict[int, LaneGroupMetrics] = {}
        self.global_metrics: Dict = {}
        
    def _preprocess_data(self):
        if self.verbose:
            print(f"  Points: {len(self.df)}")
            print(f"  Duration: {self.df['time_s'].max() - self.df['time_s'].min():.2f} s")
            print(f"  Vehicles: {self.df['vehicle_id'].nunique()}")
            print(f"  Lanes: {sorted(self.df['lane_id'].unique())}")
        
        self.df = self.df.sort_values(['vehicle_id', 'time_s']).reset_index(drop=True)
        
        self.df['dx'] = self.df.groupby('vehicle_id')['x_m'].diff()
        self.df['dy'] = self.df.groupby('vehicle_id')['y_m'].diff()
        self.df['dt'] = self.df.groupby('vehicle_id')['time_s'].diff()
        
        self.df['distance'] = np.sqrt(self.df['dx']**2 + self.df['dy']**2)
        self.df['speed_ms'] = self.df['distance'] / self.df['dt']
        self.df['speed_kmh'] = self.df['speed_ms'] * 3.6
        
        self.df['speed_ms'] = self.df['speed_ms'].fillna(0)
        self.df['speed_kmh'] = self.df['speed_kmh'].fillna(0)
        
        max_realistic_speed = 30.0
        self.df.loc[self.df['speed_ms'] > max_realistic_speed, 'speed_ms'] = np.nan
        self.df.loc[self.df['speed_kmh'] > max_realistic_speed * 3.6, 'speed_kmh'] = np.nan
        
        self.df['speed_ms'] = self.df.groupby('vehicle_id')['speed_ms'].ffill()
        self.df['speed_kmh'] = self.df.groupby('vehicle_id')['speed_kmh'].ffill()
        
        self.df['is_stopped'] = self.df['speed_ms'] < STOPPED_SPEED_THRESHOLD
        self.df['is_queued'] = self.df['speed_ms'] < QUEUE_SPEED_THRESHOLD
        
        if self.verbose:
            print(f"  Mean speed: {self.df['speed_ms'].mean():.2f} m/s ({self.df['speed_ms'].mean()*3.6:.1f} km/h)")
        
    def analyze(self, print_results=True):
        if self.verbose:
            print("\nAnalyzing...")
        self._analyze_vehicles()
        self._analyze_lane_groups()
        self._calculate_global_metrics()
        
        if print_results and self.verbose:
            self._print_results()
        
        return self.global_metrics
    
    def _analyze_vehicles(self):
        track_lengths = self.df.groupby('vehicle_id').size()
        valid_tracks = track_lengths[track_lengths >= MIN_TRACK_POINTS].index
        
        if self.verbose:
            total_vehicles = len(valid_tracks)
            print(f"  Total vehicles with >= {MIN_TRACK_POINTS} points: {total_vehicles}")
        
        vehicles_counted = 0
        vehicles_filtered_frames = 0
        vehicles_filtered_time = 0
        
        for vehicle_id in valid_tracks:
            vehicle_data = self.df[self.df['vehicle_id'] == vehicle_id].copy()
            
            # Check 1: Skip vehicles with less than MIN_FRAMES_FOR_COUNTING frames
            if len(vehicle_data) < MIN_FRAMES_FOR_COUNTING:
                vehicles_filtered_frames += 1
                continue
            
            first_seen = vehicle_data['time_s'].min()
            last_seen = vehicle_data['time_s'].max()
            total_time = last_seen - first_seen
            
            # Check 2: Skip vehicles detected for less than MIN_DETECTION_TIME seconds
            if total_time < MIN_DETECTION_TIME:
                vehicles_filtered_time += 1
                continue
            
            vehicles_counted += 1
            
            total_distance = vehicle_data['distance'].sum()
            avg_speed = vehicle_data['speed_ms'].mean()
            max_speed = vehicle_data['speed_ms'].max()
            
            # CHANGED: Calculate stopped delay (time completely stopped)
            stopped_delay = vehicle_data.loc[vehicle_data['is_stopped'], 'dt'].sum()
            
            # CHANGED: Calculate control delay (actual time - free flow time)
            # Free flow time based on distance traveled at free flow speed
            if total_distance > 0 and self.free_flow_speed_mps > 0:
                free_flow_time = total_distance / self.free_flow_speed_mps
            else:
                free_flow_time = total_time
            
            # Control delay = actual travel time - free flow time
            control_delay = max(0, total_time - free_flow_time)
            
            queue_time = vehicle_data.loc[vehicle_data['is_queued'], 'dt'].sum()
            
            is_stopped_arr = vehicle_data['is_stopped'].values
            num_stops = np.sum(np.diff(is_stopped_arr.astype(int)) > 0)
            
            # A vehicle is "completed" if it entered after measurement start time
            completed = first_seen >= MEASUREMENT_START_TIME
            
            # Assign vehicle to the lane where it spent most time
            lane_id = int(vehicle_data['lane_id'].mode()[0]) if 'lane_id' in vehicle_data.columns else 0
            
            self.vehicle_metrics[vehicle_id] = VehicleMetrics(
                vehicle_id=vehicle_id,
                lane_id=lane_id,
                first_seen=first_seen,
                last_seen=last_seen,
                total_time=total_time,
                total_distance=total_distance,
                avg_speed=avg_speed,
                max_speed=max_speed,
                stopped_delay=stopped_delay,      # Time spent stopped
                control_delay=control_delay,      # Total control delay
                queue_time=queue_time,
                free_flow_time=free_flow_time,
                completed=completed,
                num_stops=num_stops,
                trajectory_points=len(vehicle_data)
            )
        
        if self.verbose:
            print(f"  Vehicles counted (>= {MIN_FRAMES_FOR_COUNTING} frames AND >= {MIN_DETECTION_TIME}s): {vehicles_counted}")
            if vehicles_filtered_frames > 0:
                print(f"  Filtered out (< {MIN_FRAMES_FOR_COUNTING} frames): {vehicles_filtered_frames}")
            if vehicles_filtered_time > 0:
                print(f"  Filtered out (< {MIN_DETECTION_TIME}s duration): {vehicles_filtered_time}")
    
    def _analyze_lane_groups(self):
        measurement_duration = self.df['time_s'].max() - MEASUREMENT_START_TIME
        
        # Only analyze lanes with positive lane_id
        lanes = [lid for lid in self.df['lane_id'].unique() if lid >= 0]
        
        for lane_id in lanes:
            lane_vehicles = [m for m in self.vehicle_metrics.values() if m.lane_id == lane_id]
            
            if not lane_vehicles:
                continue
            
            # All unique vehicles in this lane
            n_vehicles = len(lane_vehicles)
            
            # Completed vehicles
            completed = [v for v in lane_vehicles if v.completed]
            n_completed = len(completed)
            
            # If no completed vehicles, skip this lane
            if n_completed == 0:
                continue
            
            throughput = n_vehicles
            
            # CHANGED: Calculate average control delay for ALL vehicles (not just completed)
            # This ensures we include all vehicles in the average
            avg_control_delay = np.mean([v.control_delay for v in lane_vehicles])
            
            # CHANGED: Calculate average stopped delay for ALL vehicles
            avg_stopped_delay = np.mean([v.stopped_delay for v in lane_vehicles])
            
            # Other metrics use completed vehicles
            avg_speed = np.mean([v.avg_speed for v in completed])
            
            measured_flow = (n_vehicles / measurement_duration) * 3600 if measurement_duration > 0 else 0
            
            los = calculate_LOS_from_delay(avg_control_delay)
            completion_rate = n_completed / n_vehicles if n_vehicles > 0 else 0
            
            self.lane_metrics[lane_id] = LaneGroupMetrics(
                lane_id=lane_id,
                throughput=throughput,
                n_vehicles_completed=n_completed,
                n_vehicles_total=n_vehicles,
                avg_control_delay=avg_control_delay,    # Average control delay
                avg_stopped_delay=avg_stopped_delay,    # Average stopped delay
                avg_speed=avg_speed,
                measured_flow=measured_flow,
                los=los,
                completion_rate=completion_rate
            )
    
    def _calculate_global_metrics(self):
        completed = [v for v in self.vehicle_metrics.values() if v.completed]
        all_vehicles = list(self.vehicle_metrics.values())
        
        measurement_duration = self.df['time_s'].max() - MEASUREMENT_START_TIME
        
        # Use ALL vehicles for delay calculations (not just completed)
        avg_control_delay = np.mean([v.control_delay for v in all_vehicles]) if all_vehicles else 0
        avg_stopped_delay = np.mean([v.stopped_delay for v in all_vehicles]) if all_vehicles else 0
        
        self.global_metrics = {
            'n_vehicles_total': len(all_vehicles),
            'n_vehicles_completed': len(completed),
            'completion_rate': len(completed) / len(all_vehicles) if all_vehicles else 0,
            'avg_control_delay': avg_control_delay,
            'avg_stopped_delay': avg_stopped_delay,
            'avg_speed_ms': np.mean([v.avg_speed for v in completed]) if completed else 0,
            'avg_speed_kmh': np.mean([v.avg_speed for v in completed]) * 3.6 if completed else 0,
            'total_throughput': len(completed),
            'throughput_per_hour': (len(completed) / measurement_duration * 3600) if measurement_duration > 0 else 0,
            'measurement_duration': measurement_duration,
            'overall_los': calculate_LOS_from_delay(avg_control_delay),
            'lane_metrics': {lid: vars(lm) for lid, lm in self.lane_metrics.items()}
        }
    
    def _print_results(self):
        print("\n" + "="*80)
        print("GLOBAL METRICS")
        print("="*80)
        print(f"Total vehicles: {self.global_metrics['n_vehicles_total']}")
        print(f"Completed vehicles: {self.global_metrics['n_vehicles_completed']} ({self.global_metrics['completion_rate']*100:.1f}%)")
        print(f"Average control delay: {self.global_metrics['avg_control_delay']:.2f} s/veh")
        print(f"Average stopped delay: {self.global_metrics['avg_stopped_delay']:.2f} s/veh")
        delay_ratio_global = (self.global_metrics['avg_stopped_delay'] / self.global_metrics['avg_control_delay'] * 100) if self.global_metrics['avg_control_delay'] > 0 else 0.0
        print(f"Delay ratio (stopped/control): {delay_ratio_global:.1f}%")
        print(f"Average speed: {self.global_metrics['avg_speed_ms']:.2f} m/s ({self.global_metrics['avg_speed_kmh']:.1f} km/h)")
        print(f"Throughput: {self.global_metrics['throughput_per_hour']:.1f} veh/h")
        print(f"Overall LOS: {self.global_metrics['overall_los']}")
        
        print("\n" + "="*80)
        print("LANE-BY-LANE METRICS")
        print("="*80)
        
        for lane_id, metrics in self.lane_metrics.items():
            delay_ratio = (metrics.avg_stopped_delay / metrics.avg_control_delay * 100) if metrics.avg_control_delay > 0 else 0
            print(f"\nLane {lane_id}:")
            print(f"  Vehicles: {metrics.n_vehicles_completed}/{metrics.n_vehicles_total} ({metrics.completion_rate*100:.1f}%)")
            print(f"  Avg control delay: {metrics.avg_control_delay:.2f} s/veh")
            print(f"  Avg stopped delay: {metrics.avg_stopped_delay:.2f} s/veh ({delay_ratio:.1f}% of control delay)")
            print(f"  Avg speed: {metrics.avg_speed:.2f} m/s ({metrics.avg_speed*3.6:.1f} km/h)")
            print(f"  Flow: {metrics.measured_flow:.1f} veh/h")
            print(f"  LOS: {metrics.los}")
    
    def export_csv(self, output_path: str, include_delay: bool = False, 
                   include_los: bool = False):
        """
        Export minute-by-minute lane metrics to CSV
        
        Output format:
        Minute, lane_id, n_vehicles, avg_speed_kmh [, optional metrics]
        
        Where:
        - Minute: Time interval (1, 2, 3, ...) - starts from 1
        - lane_id: Lane identifier
        - n_vehicles: NEW unique vehicles entering in this minute in THIS LANE (not counted in previous minutes in this lane)
        - avg_speed_kmh: Average speed of vehicles in this lane during this minute
        
        Optional metrics (if enabled):
        - avg_delay_s: Average control delay (actual time - free flow time)
        - los: Level of Service (A-F) based on delay
        """
        
        # Get total duration in seconds
        if len(self.df) == 0:
            total_duration = 0.0
            num_minutes = 0
        else:
            total_duration = self.df['time_s'].max() - self.df['time_s'].min()
            num_minutes = int(np.ceil(total_duration / 60))
        
        if self.verbose:
            print(f"\nGenerating minute-by-minute analysis...")
            print(f"  Total duration: {total_duration:.1f} seconds ({num_minutes} minutes)")
        
        # Get all lane IDs
        lane_ids = sorted([lid for lid in self.df['lane_id'].unique() if lid >= 0])
        
        # Track which vehicles we've counted PER LANE (separate tracking for each lane)
        counted_vehicles_per_lane = {lane_id: set() for lane_id in lane_ids}
        
        rows = []
        
        for minute_idx in range(num_minutes):
            # Minute number starts from 1 instead of 0
            minute = minute_idx + 1
            
            # Time range for this minute
            start_time = minute_idx * 60
            end_time = (minute_idx + 1) * 60
            
            # Get data for this minute
            minute_data = self.df[(self.df['time_s'] >= start_time) & 
                                  (self.df['time_s'] < end_time)].copy()
            
            if len(minute_data) == 0:
                # No data in this minute, still output 0s for all lanes
                for lane_id in lane_ids:
                    rows.append({
                        'Minute': minute,
                        'lane_id': lane_id,
                        'n_vehicles': 0,
                        'avg_speed_kmh': 0.0
                    })
                continue
            
            # Process each lane
            for lane_id in lane_ids:
                lane_minute_data = minute_data[minute_data['lane_id'] == lane_id]
                
                if len(lane_minute_data) == 0:
                    # No vehicles in this lane during this minute
                    rows.append({
                        'Minute': minute,
                        'lane_id': lane_id,
                        'n_vehicles': 0,
                        'avg_speed_kmh': 0.0
                    })
                    continue
                
                # Get unique vehicles in this lane during this minute
                vehicles_this_minute = set(lane_minute_data['vehicle_id'].unique())
                
                # NEW vehicles = vehicles not counted before IN THIS SPECIFIC LANE
                new_vehicles = vehicles_this_minute - counted_vehicles_per_lane[lane_id]
                n_new_vehicles = len(new_vehicles)
                
                # Add these vehicles to the counted set FOR THIS LANE
                counted_vehicles_per_lane[lane_id].update(new_vehicles)
                
                # Calculate average speed for vehicles in this lane this minute
                if 'speed_kmh' in lane_minute_data.columns:
                    avg_speed = lane_minute_data['speed_kmh'].mean()
                else:
                    # Calculate from speed_ms if available
                    avg_speed = lane_minute_data['speed_ms'].mean() * 3.6 if 'speed_ms' in lane_minute_data.columns else 0.0
                
                # Prepare row data
                row_data = {
                    'Minute': minute,
                    'lane_id': lane_id,
                    'n_vehicles': n_new_vehicles,
                    'avg_speed_kmh': round(avg_speed, 2)
                }
                
                # Calculate optional metrics if requested
                if include_delay or include_los:
                    # Get data for NEW vehicles only (not previously counted)
                    new_vehicles_data = lane_minute_data[lane_minute_data['vehicle_id'].isin(new_vehicles)]
                    
                    if len(new_vehicles_data) > 0 and n_new_vehicles > 0:
                        # Calculate control delay for new vehicles
                        delays = []
                        for vid in new_vehicles:
                            veh_data = new_vehicles_data[new_vehicles_data['vehicle_id'] == vid]
                            if len(veh_data) > 1:
                                # Calculate distance traveled
                                if 'distance' in veh_data.columns:
                                    distance = veh_data['distance'].sum()
                                else:
                                    coords = veh_data[['x_m', 'y_m']].values
                                    if len(coords) > 1:
                                        distance = np.sum(np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1)))
                                    else:
                                        distance = 0
                                
                                # Actual time
                                actual_time = veh_data['time_s'].max() - veh_data['time_s'].min()
                                
                                # Free flow time
                                if distance > 0 and self.free_flow_speed_mps > 0:
                                    free_flow_time = distance / self.free_flow_speed_mps
                                    delay = max(0, actual_time - free_flow_time)
                                    delays.append(delay)
                        
                        avg_delay = np.mean(delays) if delays else 0.0
                        if include_delay:
                            row_data['avg_delay_s'] = round(avg_delay, 2)
                        
                        if include_los:
                            row_data['los'] = calculate_LOS_from_delay(avg_delay)
                    else:
                        # No new vehicles, set optional metrics to 0
                        if include_delay:
                            row_data['avg_delay_s'] = 0.0
                        if include_los:
                            row_data['los'] = 'A'
                
                rows.append(row_data)
        
        # Create DataFrame and save
        df = pd.DataFrame(rows)
        if len(df) == 0:
            cols = ['Minute', 'lane_id', 'n_vehicles', 'avg_speed_kmh']
            if include_delay:
                cols.append('avg_delay_s')
            if include_los:
                cols.append('los')
            df = pd.DataFrame(columns=cols)
        df.to_csv(output_path, index=False)
        
        if self.verbose:
            print(f"\nExported minute-by-minute metrics to: {output_path}")
            print("\nCSV Columns:")
            print("  • Minute: Time interval (1, 2, 3, ...) - starts from 1")
            print("  • lane_id: Lane identifier")
            print("  • n_vehicles: NEW unique vehicles entering this minute in this lane")
            print("  • avg_speed_kmh: Average speed in this lane during this minute")
            


## Step 2: Camera Calibration & Interactive Lane Drawing

Since Colab runs on a remote server, we cannot open an interactive mouse-clicking popup window. Instead, we use a grid overlay:
1. Run the cell below to extract a frame from your video and overlay grid coordinate lines.
2. Read the image pixel coordinates from the X and Y axes.
3. Type the coordinates for the **ROI Polygon** (4 corners clockwise) and your **Lanes** (each has 4 corners clockwise) into the configuration cell, then run it to preview your drawing.

In [ ]:
# Video configuration
VIDEO_PATH = "vehicles.mp4"
WEIGHTS_PATH = "yolo11l.pt"
OUTPUT_DIR = "./output"

# Download a demo video if no video is uploaded yet
if not os.path.exists(VIDEO_PATH):
    print(f"⚠️ Video file '{VIDEO_PATH}' not found. Downloading a sample video for demonstration...")
    import urllib.request
    url = "https://media.roboflow.com/supervision/video-examples/vehicles.mp4"
    # Add User-Agent header to avoid 403 Forbidden on some CDNs
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, open('vehicles.mp4', 'wb') as out_file:
        out_file.write(response.read())
    VIDEO_PATH = "vehicles.mp4"

print(f"✅ Video: {VIDEO_PATH}")
print(f"✅ Weights: {WEIGHTS_PATH}")

In [ ]:
# Extract the first frame and display it with a pixel grid
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Could not open video.")
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w, _ = frame.shape
    
    plt.figure(figsize=(14, 9))
    plt.imshow(frame_rgb)
    # Add grid lines every 50 pixels for precise coordinate reading
    plt.grid(True, which='both', color='red', linestyle='--', linewidth=0.5)
    plt.title(f"First Frame Grid (Resolution: {w}x{h}) - Use this to read coordinates", fontsize=14, fontweight='bold')
    plt.xlabel("Image X (pixels)")
    plt.ylabel("Image Y (pixels)")
    plt.xlim(0, w)
    plt.ylim(h, 0)
    plt.show()
else:
    print("Error reading frame from video.")

In [ ]:
# CONFIGURE YOUR COORDINATES HERE (using the X/Y grid image above)

# 1. ROI Polygon coordinates (4 points: Top-Left, Top-Right, Bottom-Right, Bottom-Left)
# For the default demo video, we use these coordinates:
ROI_POLYGON = np.array([
    [540, 240],   # P1: Top-Left
    [1060, 240],  # P2: Top-Right
    [1420, 960],  # P3: Bottom-Right
    [100, 960]    # P4: Bottom-Left
], dtype=np.float32)

# Real-world dimensions of the ROI polygon (meters) for speed calibration
REAL_WORLD_WIDTH_M = 15.0   # Distance P1-P2 (meters)
REAL_WORLD_HEIGHT_M = 40.0  # Distance P2-P3 (meters)

# 2. Lanes coordinates (List of lanes, each is a 4-point polygon clockwise)
# For the default demo video, we define 2 lanes:
LANES = [
    # Lane 1 (Left lane in video)
    np.array([[550, 250], [800, 250], [700, 950], [150, 950]], dtype=np.float32),
    # Lane 2 (Right lane in video)
    np.array([[810, 250], [1050, 250], [1400, 950], [720, 950]], dtype=np.float32)
]

print("✅ Coordinates set. Run the cell below to preview the drawing!")

In [ ]:
# Preview drawing of ROI and Lanes
if ret:
    preview_img = frame_rgb.copy()
    
    # Draw ROI polygon boundary
    cv2.polylines(preview_img, [ROI_POLYGON.astype(np.int32)], True, (255, 255, 0), 4)
    for i, p in enumerate(ROI_POLYGON.astype(int)):
        cv2.circle(preview_img, tuple(p), 12, (255, 0, 0), -1)
        cv2.putText(preview_img, f"P{i+1}", (p[0]-20, p[1]-20), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)
        
    # Draw lanes boundaries
    for i, lane in enumerate(LANES):
        cv2.polylines(preview_img, [lane.astype(np.int32)], True, (0, 255, 0), 3)
        cx = int(np.mean(lane[:, 0]))
        cy = int(np.mean(lane[:, 1]))
        cv2.putText(preview_img, f"Lane {i+1}", (cx-60, cy), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        
    plt.figure(figsize=(14, 9))
    plt.imshow(preview_img)
    plt.title("Drawing Preview: Yellow is ROI boundary, Green represents Lanes", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()

    # --- Generate calibration outputs matching GUI app ---
    calib_dir = os.path.join(OUTPUT_DIR, "calib_screens")
    os.makedirs(calib_dir, exist_ok=True)
    
    # 1. Grab and save 4 snapshot frames
    cap = cv2.VideoCapture(VIDEO_PATH)
    if cap.isOpened():
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idxs = [0, max(0, total_frames // 4), max(0, total_frames // 2), max(0, 3 * total_frames // 4)]
        for idx_i, idx in enumerate(idxs):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, f = cap.read()
            if ok:
                cv2.imwrite(os.path.join(calib_dir, f"snap_{idx_i+1}_frame{idx}.png"), f)
        cap.release()
        print(f"Saved snapshot frames to {calib_dir}")
        
    # 2. Save calibration report configuration
    poly_str = f"np.array([{', '.join([str([int(x), int(y)]) for x, y in ROI_POLYGON.tolist()])}])"
    out_txt = os.path.join(calib_dir, "calibration_output.txt")
    with open(out_txt, "w", encoding="utf-8") as fh:
        fh.write(f"Video: {VIDEO_PATH}\n")
        cap = cv2.VideoCapture(VIDEO_PATH)
        if cap.isOpened():
            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            FPS = cap.get(cv2.CAP_PROP_FPS)
            fh.write(f"Frame size: {W}x{H} @ {FPS:.2f} fps\n")
            cap.release()
        fh.write("Chosen snapshot index: 0\n\n")
        fh.write("ROI Polygon (image pixels):\n")
        for i, (x, y) in enumerate(ROI_POLYGON.tolist(), 1):
            fh.write(f"  P{i}: [{int(x)}, {int(y)}]\n")
        fh.write("\nPolygon as numpy: " + poly_str + "\n\n")
        fh.write("Known real-world lengths (meters):\n")
        fh.write(f"  L12 (P1-P2): {REAL_WORLD_WIDTH_M:.6f}\n")
        fh.write(f"  L23 (P2-P3): {REAL_WORLD_HEIGHT_M:.6f}\n\n")
        fh.write(f"Number of lanes: {len(LANES)}\n")
        for i, lane in enumerate(LANES, start=1):
            fh.write(f"\nLane {i} (image pixels):\n")
            for j, (x, y) in enumerate(lane.tolist(), 1):
                fh.write(f"  P{j}: [{int(x)}, {int(y)}]\n")
    print(f"Saved calibration report to {out_txt}")
    
    # 3. Save calibration visual overlay (in BGR for cv2)
    vis = frame.copy()
    overlay = vis.copy()
    cv2.fillPoly(overlay, [ROI_POLYGON.astype(np.int32)], (128, 128, 128))
    cv2.addWeighted(overlay, 0.25, vis, 0.75, 0, vis)
    cv2.polylines(vis, [ROI_POLYGON.astype(np.int32)], True, (0, 255, 255), 2)
    for i, p in enumerate(ROI_POLYGON.astype(int).tolist(), 1):
        cv2.circle(vis, (p[0], p[1]), 6, (0, 255, 255), -1)
        cv2.putText(vis, str(i), (p[0] + 6, p[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
    for lane_idx, lane in enumerate(LANES):
        color = LANE_COLORS[lane_idx % len(LANE_COLORS)]
        lane_pts = lane.astype(np.int32)
        overlay = vis.copy()
        cv2.fillPoly(overlay, [lane_pts], color)
        cv2.addWeighted(overlay, 0.3, vis, 0.7, 0, vis)
        cv2.polylines(vis, [lane_pts], True, color, 2)
        cx = int(np.mean(lane[:, 0]))
        cy = int(np.mean(lane[:, 1]))
        cv2.putText(vis, f"Lane {lane_idx + 1}", (cx - 30, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.imwrite(os.path.join(calib_dir, "calibration_visual.png"), vis)
    print(f"Saved calibration visual to {os.path.join(calib_dir, 'calibration_visual.png')}")

## Step 3: Run YOLO Detection & ByteTrack Tracking

This cell executes the detection and tracking. It instantiates the YOLO model, applies ByteTrack inside the ROI, uses the homography matrix to project points for speed calculations, and saves coordinates to `vehicle_tracks_xy.csv`.

In [ ]:
import os
import sys
import csv
import logging
import inspect
from collections import defaultdict, deque, Counter
import supervision as sv
from ultralytics import YOLO
import numpy as np
import cv2

# --- Configuration Parameters (matches GUI app) ---
CONFIDENCE_THRESHOLD = 0.5
NMS_IOU = 0.5
IMGSZ = 1280
BLUR_OUTSIDE = True
FORCE_CPU = False
VEHICLES_ONLY = True  # True: Filters COCO vehicle classes [2,3,5,7]. False: Tracks all classes like the app.

os.makedirs(OUTPUT_DIR, exist_ok=True)
csv_lanes_path = os.path.join(OUTPUT_DIR, "lanes.csv")
csv_summary_path = os.path.join(OUTPUT_DIR, "vehicles.csv")
csv_tracks_path = os.path.join(OUTPUT_DIR, "vehicle_tracks_xy.csv")
target_video_path = os.path.join(OUTPUT_DIR, "vehicles-result.mp4")

# Set up logging to stdout and runtime.log (mirrors GUI app)
logger = logging.getLogger()
for h in list(logger.handlers):
    logger.removeHandler(h)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(OUTPUT_DIR, "runtime.log"), mode="w", encoding="utf-8"),
        logging.StreamHandler(sys.stdout)
    ]
)

logging.info("Starting detection and tracking pipeline...")

# 1. Save lanes to CSV
with open(csv_lanes_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["lane_id", "p1_x", "p1_y", "p2_x", "p2_y", "p3_x", "p3_y", "p4_x", "p4_y"])
    for i, lane in enumerate(LANES, start=1):
        pts = lane.flatten().tolist()
        writer.writerow([i] + [int(p) for p in pts])
logging.info(f"Saved lanes configuration to {csv_lanes_path}")

# 2. Warm up YOLO on proper device and get video properties
try:
    import torch
    DEVICE = "cpu" if FORCE_CPU else ("cuda" if torch.cuda.is_available() else "cpu")
except Exception:
    DEVICE = "cpu"
logging.info(f"Running tracking on device: {DEVICE} (force_cpu={FORCE_CPU})")

yolo = YOLO(WEIGHTS_PATH)
vidmeta = sv.VideoInfo.from_video_path(video_path=VIDEO_PATH)
frames = sv.get_video_frames_generator(source_path=VIDEO_PATH)
logging.info(f"Loaded video: {vidmeta.resolution_wh[0]}x{vidmeta.resolution_wh[1]} @ {vidmeta.fps:.2f} FPS")

# Warm up model
dummy = np.zeros((vidmeta.resolution_wh[1], vidmeta.resolution_wh[0], 3), dtype=np.uint8)
_ = yolo(dummy, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
logging.info("Warming up model... done.")

# Initialize tracker (ByteTrack) with robust version check (matches calib_and_track_ui.py)
init_params = set(inspect.signature(sv.ByteTrack.__init__).parameters.keys())
try:
    if {"track_thresh", "match_thresh", "track_buffer"} <= init_params:
        tracker = sv.ByteTrack(frame_rate=vidmeta.fps, track_thresh=0.25, match_thresh=0.6, track_buffer=int(vidmeta.fps * 8))
    elif {"track_activation_threshold"} <= init_params:
        tracker = sv.ByteTrack(frame_rate=vidmeta.fps, track_activation_threshold=CONFIDENCE_THRESHOLD)
    else:
        tracker = sv.ByteTrack(frame_rate=vidmeta.fps)
except TypeError:
    tracker = sv.ByteTrack(frame_rate=vidmeta.fps, track_activation_threshold=CONFIDENCE_THRESHOLD)

# Plane mapping for real-world coordinate mapping
CAL_DST = np.array([[0, 0], [REAL_WORLD_WIDTH_M, 0], [REAL_WORLD_WIDTH_M, REAL_WORLD_HEIGHT_M], [0, REAL_WORLD_HEIGHT_M]], dtype=np.float32)
mapper = PlaneMapper(src_quad=ROI_POLYGON, dst_quad=CAL_DST)

# Prepare file writers
csv_sum_fh = open(csv_summary_path, "w", newline="", encoding="utf-8")
csv_sum_out = csv.writer(csv_sum_fh)
csv_sum_out.writerow(["vehicle_id", "label", "avg_speed_kmh", "start_frame", "end_frame", "start_time_s", "end_time_s"])

csv_trk_fh = open(csv_tracks_path, "w", newline="", encoding="utf-8")
csv_trk_out = csv.writer(csv_trk_fh)
csv_trk_out.writerow(["frame", "time_s", "vehicle_id", "x_m", "y_m", "img_x", "img_y"])

trails_roi = defaultdict(lambda: deque(maxlen=int(vidmeta.fps)))
class_votes = defaultdict(Counter)
first_frame = {}
last_frame = {}
stationary_tracks = {}

STATIONARY_THRESHOLD = 2.0
STATIONARY_GRACE = int(vidmeta.fps * 15)
MOVING_GRACE = int(max(1, 0.5 * vidmeta.fps))

# Annotator configuration matching GUI app (optimal scale & dynamic trails)
ln_px = sv.calculate_optimal_line_thickness(resolution_wh=vidmeta.resolution_wh)
base_txt_scale = sv.calculate_optimal_text_scale(resolution_wh=vidmeta.resolution_wh)
txt_scale = base_txt_scale * 0.6
draw_boxes = sv.BoxAnnotator(thickness=ln_px)
draw_labels = sv.LabelAnnotator(text_scale=txt_scale, text_thickness=max(1, ln_px - 1), text_position=sv.Position.TOP_CENTER)
draw_traces = sv.TraceAnnotator(thickness=ln_px, trace_length=int(vidmeta.fps * 2), position=sv.Position.CENTER)

# Run tracking loop
f_idx = 0
logging.info("Processing video frames (tracking)... This may take a minute...")

with sv.VideoSink(target_video_path, vidmeta) as vsink:
    for frame in frames:
        yout = yolo(frame, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        raw_dets = sv.Detections.from_ultralytics(yout)
        dets = raw_dets
        
        # Optionally filter for vehicles only
        if VEHICLES_ONLY:
            vehicle_classes = [2, 3, 5, 7]
            dets = dets[np.isin(dets.class_id, vehicle_classes)]
            
        if dets.confidence is not None:
            dets = dets[dets.confidence > 0.15]
        
        # NMS with compatibility checks
        try:
            dets = dets.with_nms(threshold=NMS_IOU, class_agnostic=True)
        except TypeError:
            dets = dets.with_nms(threshold=NMS_IOU)
        
        # Filter to ROI boundary
        dets_roi, _ = filter_detections_to_polygon(dets, ROI_POLYGON, anchor=sv.Position.CENTER)
        
        # Tracker update
        dets_tracked = tracker.update_with_detections(detections=dets_roi)
        dets_tracked = suppress_track_dupes(dets_tracked, iou=0.85)
        
        bc_xy = dets_tracked.get_anchors_coordinates(anchor=sv.Position.CENTER)
        bc_warp = mapper.warp_points(pts_xy=bc_xy).astype(float)
        
        class_names = getattr(yout, "names", None)
        def id_to_name(c):
            if class_names and c is not None and int(c) in class_names:
                return class_names[int(c)]
            return str(int(c)) if c is not None else "unknown"
            
        ids = dets_tracked.tracker_id if dets_tracked.tracker_id is not None else np.empty((0,), dtype=int)
        cls = dets_tracked.class_id
        
        for idx in range(len(ids)):
            tid = ids[idx]
            if tid is None:
                continue
            x_m, y_m = float(bc_warp[idx][0]), float(bc_warp[idx][1])
            img_x, img_y = float(bc_xy[idx][0]), float(bc_xy[idx][1])
            
            # Speed stationary detection handling
            if tid in stationary_tracks:
                prev_pos = stationary_tracks[tid]['last_pos']
                dist_moved = np.hypot(x_m - prev_pos[0], y_m - prev_pos[1])
                if dist_moved < STATIONARY_THRESHOLD:
                    stationary_tracks[tid]['frames_stationary'] += 1
                else:
                    stationary_tracks[tid]['frames_stationary'] = 0
                    stationary_tracks[tid]['last_pos'] = (x_m, y_m)
                stationary_tracks[tid]['last_seen'] = f_idx
            else:
                stationary_tracks[tid] = {'last_pos': (x_m, y_m), 'frames_stationary': 0, 'last_seen': f_idx}
                
            trails_roi[tid].append((x_m, y_m))
            last_frame[tid] = f_idx
            if tid not in first_frame:
                first_frame[tid] = f_idx
                
            cid = None
            if cls is not None and idx < len(cls):
                cid = cls[idx]
            class_votes[tid][id_to_name(cid)] += 1
            
            t_s = f_idx / vidmeta.fps
            csv_trk_out.writerow([f_idx, round(t_s, 3), tid, round(x_m, 3), round(y_m, 3), round(img_x, 1), round(img_y, 1)])
            
        # Annotate and save frame
        lbls = []
        for tid in ids:
            pts_roi = trails_roi.get(tid, deque())
            v = mean_speed_kmh(pts_roi, vidmeta.fps)
            lbls.append(f"#{tid}" if v is None else f"#{tid} {int(v)} km/h")
            
        # Draw base stylization and box, trace, label overlays
        annotated_frame, _ = stylize_display(frame, ROI_POLYGON, LANES, ln_px, blur_outside=BLUR_OUTSIDE)
        annotated_frame = draw_traces.annotate(scene=annotated_frame, detections=dets_tracked)
        annotated_frame = draw_boxes.annotate(scene=annotated_frame, detections=dets_tracked)
        annotated_frame = draw_labels.annotate(scene=annotated_frame, detections=dets_tracked, labels=lbls)
        vsink.write_frame(annotated_frame)
        
        # Handle closing of track details
        to_close = []
        for tid in list(last_frame.keys()):
            frames_missing = f_idx - last_frame.get(tid, 0)
            grace = STATIONARY_GRACE if (tid in stationary_tracks and stationary_tracks[tid]['frames_stationary'] > vidmeta.fps) else MOVING_GRACE
            if frames_missing > grace:
                to_close.append(tid)
                
        for tid in to_close:
            pts = trails_roi.get(tid, deque())
            if len(pts) >= 2:
                v = mean_speed_kmh(pts, vidmeta.fps)
                f0 = first_frame.get(tid, 0)
                f1 = last_frame.get(tid, f_idx)
                t0 = f0 / vidmeta.fps
                t1 = f1 / vidmeta.fps
                
                votes = class_votes.get(tid, Counter())
                if len(votes) == 0:
                    label = "unknown"
                else:
                    # Choose label with highest vote excluding 'unknown'
                    label, _ = max(((k, c) for k, c in votes.items() if k != "unknown"), default=max(votes.items(), key=lambda kv: kv[1]))
                    
                if v is not None:
                    csv_sum_out.writerow([tid, label, round(v, 1), f0, f1, round(t0, 3), round(t1, 3)])
                    
            trails_roi.pop(tid, None)
            class_votes.pop(tid, None)
            first_frame.pop(tid, None)
            last_frame.pop(tid, None)
            stationary_tracks.pop(tid, None)
            
        f_idx += 1
        if f_idx % 100 == 0:
            logging.info(f"Processed {f_idx} frames...")

csv_sum_fh.close()
csv_trk_fh.close()
logging.info(f"Complete! Generated video: {target_video_path}")
logging.info(f"Generated tracks: {csv_tracks_path}")

## Step 4: Run Lane Assignment & Performance Analysis

Map every trajectory point to a defined lane ID and execute the performance metrics analyzer.

In [ ]:
# --- Configuration Parameters (matches GUI app checkboxes) ---
INCLUDE_DELAY = True
INCLUDE_LOS = True
FREE_FLOW_SPEED_KMH = 50.0

# 1. Load lanes coordinates from lanes.csv
lanes_dict = {}
with open(csv_lanes_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        lane_id = int(row['lane_id'])
        lanes_dict[lane_id] = [
            [float(row['p1_x']), float(row['p1_y'])],
            [float(row['p2_x']), float(row['p2_y'])],
            [float(row['p3_x']), float(row['p3_y'])],
            [float(row['p4_x']), float(row['p4_y'])]
        ]

# 2. Assign lane ID to each point in trajectory data
tracks_df = pd.read_csv(csv_tracks_path)
lane_ids = []
for _, row in tracks_df.iterrows():
    lane_id = assign_lane_to_point(row['img_x'], row['img_y'], lanes_dict)
    lane_ids.append(lane_id if lane_id is not None else -1)
tracks_df['lane_id'] = lane_ids

# Filter out points outside defined lanes
tracks_df = tracks_df[tracks_df['lane_id'] >= 0].copy()
csv_assigned_tracks = os.path.join(OUTPUT_DIR, "vehicle_tracks_with_lanes.csv")
tracks_df.to_csv(csv_assigned_tracks, index=False)
print(f"Assigned lanes to points. Saved to {csv_assigned_tracks}")

# 3. Run analyzer
analyzer = CVIntersectionAnalyzer(
    trajectory_file=csv_assigned_tracks,
    free_flow_speed_kmh=FREE_FLOW_SPEED_KMH,
    verbose=True
)
analyzer.lane_polygons = lanes_dict
global_metrics = analyzer.analyze(print_results=True)

# 4. Export minute-by-minute metrics CSV
csv_metrics_file = os.path.join(OUTPUT_DIR, "lane_metrics.csv")
analyzer.export_csv(
    output_path=csv_metrics_file,
    include_delay=INCLUDE_DELAY,
    include_los=INCLUDE_LOS
)

# 5. Export cv_metrics.json (matches GUI performance analysis output)
json_path = os.path.join(OUTPUT_DIR, "cv_metrics.json")
with open(json_path, 'w') as f:
    metrics_converted = convert_to_python_types(global_metrics)
    json.dump(metrics_converted, f, indent=2)
print(f"Exported global metrics to JSON: {json_path}")

# 6. Export analysis_summary.txt (matches GUI performance analysis output)
summary_file = os.path.join(OUTPUT_DIR, "analysis_summary.txt")
with open(summary_file, 'w') as f:
    f.write("INTERSECTION PERFORMANCE ANALYSIS\n")
    f.write("=" * 70 + "\n\n")
    
    import dataclasses
    for lane_id, metrics in analyzer.lane_metrics.items():
        f.write(f"\nLane {lane_id}\n")
        f.write("-" * 40 + "\n")
        if dataclasses.is_dataclass(metrics):
            metrics_dict = dataclasses.asdict(metrics)
            for key, value in metrics_dict.items():
                f.write(f"  {key}: {value}\n")
    
    if hasattr(analyzer, 'global_metrics') and analyzer.global_metrics:
        f.write(f"\n\nGLOBAL METRICS\n")
        f.write("-" * 40 + "\n")
        for key, value in analyzer.global_metrics.items():
            f.write(f"  {key}: {value}\n")
print(f"Exported text summary report: {summary_file}")

print("\n--- Sample output of exported lane_metrics.csv ---")
df_metrics = pd.read_csv(csv_metrics_file)
display(df_metrics)

## Step 5: Visualize Traffic Performance Trends

Plot charts showing traffic throughput and average speed changes over the minutes.

In [ ]:
df_metrics = pd.read_csv(csv_metrics_file)
unique_lanes = df_metrics['lane_id'].unique()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot vehicle counts
for lane_id in unique_lanes:
    lane_data = df_metrics[df_metrics['lane_id'] == lane_id]
    ax1.plot(lane_data['Minute'], lane_data['n_vehicles'], '-o', linewidth=2, label=f'Lane {lane_id}')
ax1.set_title("New Vehicles per Minute", fontsize=12, fontweight='bold')
ax1.set_xlabel("Minute")
ax1.set_ylabel("Number of Vehicles")
ax1.set_xticks(df_metrics['Minute'].unique())
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

# Plot average speeds
for lane_id in unique_lanes:
    lane_data = df_metrics[df_metrics['lane_id'] == lane_id]
    ax2.plot(lane_data['Minute'], lane_data['avg_speed_kmh'], '-s', linewidth=2, label=f'Lane {lane_id}')
ax2.set_title("Average Speed per Minute", fontsize=12, fontweight='bold')
ax2.set_xlabel("Minute")
ax2.set_ylabel("Speed (km/h)")
ax2.set_xticks(df_metrics['Minute'].unique())
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()